# NARR-to-PRISM: Random 10-Date CPU Inference

Run the production tiled-inference path for 10 reproducibly sampled dates and plot the spatial `ppt`, `tmax`, and `tmin` fields. This notebook deliberately uses CPU only and writes to a separate smoke-test directory.

> **Checkpoint requirement:** use a checkpoint trained from scratch with the corrected preprocessing/model contract. Legacy checkpoints are rejected. California-domain CPU inference can be slow; reduce `SAMPLE_COUNT` temporarily while debugging.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Run this notebook from inside the granite-wxc repository")

NARR_PRISM_DIR = REPO_ROOT / "examples" / "NARR_PRISM"
for path in (REPO_ROOT, NARR_PRISM_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Hide CUDA before importing torch. The explicit CPU device below is the
# authoritative safeguard even if torch was imported earlier in the kernel.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.chdir(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")

In [ ]:
from copy import deepcopy
from datetime import timedelta
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from granitewxc.utils.config import get_config
from granitewxc.utils.normalization import case_preprocess_dir
from narr_prism_utils import get_case_name, load_yaml, parse_date_range_from_config
import narr_prism_inference as npi

npi = reload(npi)
device = torch.device("cpu")
CPU_THREADS = min(8, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
print(f"Device: {device}; torch CPU threads: {torch.get_num_threads()}")

## Configuration

In [ ]:
# ===================== USER PARAMETERS (EDIT ME) =====================
CONFIG_PATH = NARR_PRISM_DIR / "NARR_PRISM_subdomain.yaml"
CHECKPOINT_PATH = None  # Prefer an explicit newly retrained checkpoint.
OUTPUT_ROOT = NARR_PRISM_DIR / "experiments" / "inference_random10_cpu"
SAMPLE_COUNT = 10
RANDOM_SEED = 42
BATCH_SIZE = 1
PLOT_VARIABLES = ["ppt", "tmax", "tmin"]
# ====================================================================

config_path = str(CONFIG_PATH.resolve())
cfg = deepcopy(load_yaml(config_path))
config = get_config(config_path)
case_name = get_case_name(cfg)

# Keep this smoke test small: CPU, one tile per forward call, and no large
# intermediate diagnostic archive. The normal inference correctness checks
# and overlap metrics still run.
inference_cfg = cfg.setdefault("inference", {})
inference_cfg["batch_size"] = BATCH_SIZE
inference_cfg.setdefault("diagnostics", {})["enabled"] = False

checkpoint_path = str(Path(npi._find_checkpoint(cfg, CHECKPOINT_PATH)).resolve())
missing_variables = [name for name in PLOT_VARIABLES if name not in cfg["data"]["target_variables"]]
if missing_variables:
    raise ValueError(f"Configured targets do not include: {missing_variables}")

print(f"Config: {config_path}")
print(f"Case: {case_name}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Output root: {OUTPUT_ROOT}")

## Select 10 Random Dates

The seed makes the sample reproducible. Dates are sorted after sampling so the plot reads chronologically.

In [ ]:
start_date, end_date = parse_date_range_from_config(cfg, "inference")
all_dates = [
    start_date + timedelta(days=offset)
    for offset in range((end_date - start_date).days + 1)
]
if len(all_dates) < SAMPLE_COUNT:
    raise ValueError(f"Requested {SAMPLE_COUNT} samples from only {len(all_dates)} inference dates")

rng = np.random.default_rng(RANDOM_SEED)
sample_indices = rng.choice(len(all_dates), size=SAMPLE_COUNT, replace=False)
selected_dates = sorted(all_dates[int(index)] for index in sample_indices)

print(f"Random seed: {RANDOM_SEED}")
for index, sample_date in enumerate(selected_dates, start=1):
    print(f"  {index:2d}. {sample_date.isoformat()}")

## Run CPU Inference

The checkpoint is loaded once, then the same halo, overlap, and Hann-weighted stitching used by production inference is applied to the 10 selected dates.

In [ ]:
output_path = npi.run_inference(
    config_path=config_path,
    cfg=cfg,
    config=config,
    checkpoint_path=checkpoint_path,
    output_dir=str(OUTPUT_ROOT),
    device=device,
    batch_size=BATCH_SIZE,
    show_progress=True,
    data_parallel=False,
    selected_dates=selected_dates,
)
print(f"Inference outputs: {output_path}")

## Plot the Three Spatial Fields

Each row is one sampled date. Color limits are shared down each variable column, using robust percentiles across all 10 predictions.

In [ ]:
output_path = Path(output_path)
expected_files = [
    output_path / f"{case_name}_inference_{sample_date:%Y%m%d}.nc"
    for sample_date in selected_dates
]
missing_files = [str(path) for path in expected_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing selected inference outputs:\n" + "\n".join(missing_files))

daily_datasets = []
for path in expected_files:
    with xr.open_dataset(path) as source:
        daily_datasets.append(source[PLOT_VARIABLES].load())
sample_ds = xr.concat(daily_datasets, dim="time")

# Read the date-matched PRISM targets from the canonical preprocessed daily
# products. Exact coordinate equality prevents a shifted or interpolated
# truth field from manufacturing artificial difference boundaries.
truth_root = case_preprocess_dir(cfg) / "inference"
truth_by_variable = {variable: [] for variable in PLOT_VARIABLES}
for sample_date in selected_dates:
    truth_path = truth_root / f"narr_prism_{sample_date:%Y%m%d}.nc"
    if not truth_path.exists():
        raise FileNotFoundError(f"Missing PRISM target product: {truth_path}")
    with xr.open_dataset(truth_path) as truth_source:
        np.testing.assert_array_equal(truth_source.lat.values, sample_ds.lat.values)
        np.testing.assert_array_equal(truth_source.lon.values, sample_ds.lon.values)
        for variable in PLOT_VARIABLES:
            target_name = f"target_{variable}"
            if target_name not in truth_source:
                raise KeyError(f"{truth_path} does not contain {target_name}")
            truth_by_variable[variable].append(
                np.asarray(truth_source[target_name].values, dtype=np.float32)
            )
truth_by_variable = {
    variable: np.stack(fields) for variable, fields in truth_by_variable.items()
}
print(sample_ds)
print(f"Loaded exact-grid PRISM targets from: {truth_root}")

In [ ]:
plot_styles = {
    "ppt": {"cmap": "Blues", "percentiles": (0.0, 99.0)},
    "tmax": {"cmap": "inferno", "percentiles": (1.0, 99.0)},
    "tmin": {"cmap": "inferno", "percentiles": (1.0, 99.0)},
}

# One figure per variable keeps the difference panels large enough to reveal
# native-grid or tile-grid rectangles. pcolormesh renders the actual cells;
# no smoothing or image interpolation is applied.
for variable in PLOT_VARIABLES:
    inference_values = np.asarray(sample_ds[variable].values, dtype=np.float32)
    prism_values = truth_by_variable[variable]
    difference_values = inference_values - prism_values

    field_finite = np.concatenate(
        [inference_values[np.isfinite(inference_values)], prism_values[np.isfinite(prism_values)]]
    )
    difference_finite = np.abs(difference_values[np.isfinite(difference_values)])
    if field_finite.size == 0 or difference_finite.size == 0:
        raise ValueError(f"No finite inference/PRISM comparison values for {variable}")

    low_pct, high_pct = plot_styles[variable]["percentiles"]
    field_vmin, field_vmax = np.nanpercentile(field_finite, [low_pct, high_pct])
    if variable == "ppt":
        field_vmin = 0.0
    if not np.isfinite(field_vmax) or field_vmax <= field_vmin:
        field_vmax = field_vmin + 1.0
    difference_limit = float(np.nanpercentile(difference_finite, 99.0))
    if not np.isfinite(difference_limit) or difference_limit <= 0.0:
        difference_limit = 1.0

    fig, axes = plt.subplots(
        SAMPLE_COUNT, 3, figsize=(18, 3.3 * SAMPLE_COUNT),
        sharex=True, sharey=True, squeeze=False,
    )
    field_mesh = difference_mesh = None
    for row, sample_date in enumerate(selected_dates):
        field_mesh = axes[row, 0].pcolormesh(
            sample_ds.lon, sample_ds.lat, inference_values[row], shading="auto",
            cmap=plot_styles[variable]["cmap"], vmin=float(field_vmin),
            vmax=float(field_vmax), rasterized=True,
        )
        axes[row, 1].pcolormesh(
            sample_ds.lon, sample_ds.lat, prism_values[row], shading="auto",
            cmap=plot_styles[variable]["cmap"], vmin=float(field_vmin),
            vmax=float(field_vmax), rasterized=True,
        )
        difference_mesh = axes[row, 2].pcolormesh(
            sample_ds.lon, sample_ds.lat, difference_values[row], shading="auto",
            cmap="RdBu_r", vmin=-difference_limit, vmax=difference_limit,
            rasterized=True,
        )
        axes[row, 0].set_ylabel(f"{sample_date.isoformat()}\nLatitude")
        if row == SAMPLE_COUNT - 1:
            for ax in axes[row]:
                ax.set_xlabel("Longitude")

    axes[0, 0].set_title("Inference")
    axes[0, 1].set_title("PRISM")
    axes[0, 2].set_title("Inference − PRISM")
    units = sample_ds[variable].attrs.get("units", "")
    fig.colorbar(
        field_mesh, ax=axes[:, :2].ravel().tolist(), fraction=0.012, pad=0.012,
        label=units,
    )
    fig.colorbar(
        difference_mesh, ax=axes[:, 2].tolist(), fraction=0.024, pad=0.012,
        label=units,
    )
    fig.suptitle(
        f"{variable}: {SAMPLE_COUNT} random CPU inference dates "
        f"(seed={RANDOM_SEED}; difference scale ±{difference_limit:.3g} {units})",
        fontsize=15, y=0.995,
    )
    fig.subplots_adjust(top=0.975, hspace=0.12, wspace=0.06, right=0.94)
    plt.show()

In [ ]:
sample_ds.close()
print("Done.")